# مشروع تصنيف أمراض الجلد باستخدام HAM10000 وEfficientNetB0

**دفتر مختصر ومرتب للتنفيذ والتسليم – Google Colab**

هذا الدفتر يحتفظ بالخطوات الأساسية للمشروع فقط: تجهيز البيانات، تقسيمها دون تسرب صور الآفة الواحدة، تدريب EfficientNetB0 بحجم 260×260، التقييم النهائي، وحفظ الملفات مباشرة في Google Drive.

> **تنبيه:** المشروع تعليمي وبحثي، ولا يُستخدم لتشخيص الحالات أو اتخاذ قرارات علاجية.

## تسلسل العمل
1. تهيئة بيئة Colab وربط Google Drive.
2. تنزيل HAM10000 والتحقق من سلامة الملفات.
3. قراءة البيانات وربط الصور بالفئات.
4. تقسيم البيانات باستخدام `StratifiedGroupKFold`.
5. تجهيز الصور بحجم 260×260 وتحسين بيانات التدريب.
6. بناء EfficientNetB0 والتدريب الأولي.
7. Fine-Tuning وحفظ أفضل الأوزان في Drive.
8. التقييم النهائي باستخدام TTA.
9. حفظ النموذج والنتائج والتوثيق في Drive.

## 1. تهيئة البيئة ومجلد الحفظ

فعّلي GPU من: **Runtime → Change runtime type → GPU**.

سنحفظ النماذج والنتائج في Google Drive حتى لا تضيع عند انتهاء جلسة Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

SAVE_DIR = "/content/drive/MyDrive/Skin_Project_Final"
os.makedirs(SAVE_DIR, exist_ok=True)

print("مجلد حفظ المشروع:", SAVE_DIR)

## 2. تنزيل بيانات HAM10000 والتحقق منها

تنزّل هذه الخطوة ملف الصور وملف البيانات الوصفية من Zenodo، وتتحقق من سلامة التنزيل باستخدام MD5 قبل فك الضغط.
قد يستغرق تنزيل الصور وقتًا بسبب حجمها.

In [ ]:
# ============================================================
# HAM10000 - Complete Setup, Download, Verification and Extraction
# تحميل وتجهيز بيانات HAM10000 بالكامل من Zenodo
# ============================================================

import os
import sys
import json
import time
import hashlib
import zipfile
import subprocess
import requests
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. تثبيت المكتبات والأدوات اللازمة
# ------------------------------------------------------------

print("تجهيز بيئة العمل...")

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "aria2"],
    check=False
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "scikit-learn", "seaborn", "tqdm"],
    check=True
)

# ------------------------------------------------------------
# 2. تحديد المجلدات والمسارات
# ------------------------------------------------------------

DATA_ROOT = Path("/content/ham10000")
DOWNLOAD_DIR = DATA_ROOT / "downloads"
IMAGE_DIR = DATA_ROOT / "images"

ZIP_PATH = DOWNLOAD_DIR / "HAM10000_all_images.zip"
METADATA_PATH = DATA_ROOT / "HAM10000_metadata.csv"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. الحصول على روابط الملفات وبيانات التحقق من Zenodo
# ------------------------------------------------------------

RECORD_ID = "17702692"

API_URL = f"https://zenodo.org/api/records/{RECORD_ID}"

print("\nالاتصال بمصدر البيانات...")

response = requests.get(API_URL, timeout=60)
response.raise_for_status()

record = response.json()

files_info = {
    item["key"]: item
    for item in record["files"]
}

IMAGE_FILENAME = "HAM10000_all_images.zip"
CSV_FILENAME = "HAM10000_metadata.csv"

if IMAGE_FILENAME not in files_info or CSV_FILENAME not in files_info:
    raise RuntimeError(
        "لم يتم العثور على ملفات HAM10000 المطلوبة في سجل Zenodo."
    )

image_info = files_info[IMAGE_FILENAME]
csv_info = files_info[CSV_FILENAME]

IMAGE_URL = image_info["links"]["self"]
METADATA_URL = csv_info["links"]["self"]

IMAGE_MD5 = image_info["checksum"].replace("md5:", "")
CSV_MD5 = csv_info["checksum"].replace("md5:", "")

IMAGE_SIZE = int(image_info["size"])
CSV_SIZE = int(csv_info["size"])

print("تم العثور على الملفات:")
print(f"ملف الصور: {IMAGE_SIZE / (1024**3):.2f} GB")
print(f"ملف البيانات: {CSV_SIZE / 1024:.1f} KB")

# ------------------------------------------------------------
# 4. حساب MD5 للتحقق من سلامة الملفات
# ------------------------------------------------------------

def calculate_md5(file_path):
    md5 = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            md5.update(chunk)

    return md5.hexdigest()


# ------------------------------------------------------------
# 5. التحقق من الملفات الموجودة مسبقاً
# ------------------------------------------------------------

def is_valid_file(path, expected_size, expected_md5):
    path = Path(path)

    if not path.exists():
        return False

    if path.stat().st_size != expected_size:
        return False

    print(f"\nالتحقق من الملف: {path.name}")

    actual_md5 = calculate_md5(path)

    return actual_md5.lower() == expected_md5.lower()


# ------------------------------------------------------------
# 6. تنزيل الملفات باستخدام aria2
#    يدعم استكمال التنزيل عند انقطاع الاتصال
# ------------------------------------------------------------

def download_file(url, destination, expected_size, expected_md5):

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if is_valid_file(destination, expected_size, expected_md5):
        print(f"الملف مكتمل وسليم: {destination.name}")
        return

    if destination.exists():
        print("الملف الموجود غير مكتمل أو غير سليم.")
        print("سيتم محاولة استكمال التنزيل.")

    print(f"\nبدء تنزيل: {destination.name}")

    aria2_available = (
        subprocess.run(
            ["bash", "-lc", "command -v aria2c"],
            capture_output=True
        ).returncode == 0
    )

    if aria2_available:

        command = [
            "aria2c",
            "--continue=true",
            "--max-connection-per-server=8",
            "--split=8",
            "--min-split-size=5M",
            "--max-tries=20",
            "--retry-wait=5",
            "--connect-timeout=30",
            "--timeout=120",
            "--file-allocation=none",
            "--allow-overwrite=true",
            "--auto-file-renaming=false",
            "--check-integrity=true",
            f"--checksum=md5={expected_md5}",
            "--summary-interval=10",
            "--console-log-level=notice",
            "--dir", str(destination.parent),
            "--out", destination.name,
            url
        ]

        subprocess.run(command, check=True)

    else:

        print("aria2 غير متاح، سيتم استخدام التنزيل البديل.")

        headers = {}
        downloaded = 0

        if destination.exists():
            downloaded = destination.stat().st_size

        if downloaded >= expected_size:
            downloaded = 0

        if downloaded > 0:
            headers["Range"] = f"bytes={downloaded}-"

        with requests.get(
            url,
            headers=headers,
            stream=True,
            timeout=(30, 180)
        ) as r:

            r.raise_for_status()

            if downloaded > 0 and r.status_code != 206:
                downloaded = 0

            mode = "ab" if downloaded > 0 else "wb"

            with open(destination, mode) as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)

    if not is_valid_file(destination, expected_size, expected_md5):
        raise RuntimeError(
            f"فشل التحقق من الملف {destination.name}. "
            "أعيدي تشغيل الخلية لمحاولة استكمال التنزيل."
        )

    print(f"تم تنزيل الملف والتحقق منه: {destination.name}")


# ------------------------------------------------------------
# 7. تنزيل الصور والبيانات الوصفية
# ------------------------------------------------------------

download_file(
    IMAGE_URL,
    ZIP_PATH,
    IMAGE_SIZE,
    IMAGE_MD5
)

download_file(
    METADATA_URL,
    METADATA_PATH,
    CSV_SIZE,
    CSV_MD5
)

print("\nاكتمل تنزيل الملفات والتحقق منها.")

# ------------------------------------------------------------
# 8. فك ضغط الصور
# ------------------------------------------------------------

print("\nفحص الصور الموجودة...")

existing_images = list(IMAGE_DIR.rglob("*.jpg"))
existing_images += list(IMAGE_DIR.rglob("*.jpeg"))
existing_images += list(IMAGE_DIR.rglob("*.png"))

if len(existing_images) >= 10000:

    print(f"الصور موجودة بالفعل: {len(existing_images)} صورة.")

else:

    print("بدء فك ضغط ملف الصور...")

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:

        bad_file = archive.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"ملف ZIP يحتوي على ملف تالف: {bad_file}"
            )

        archive.extractall(IMAGE_DIR)

    print("اكتمل فك الضغط.")

# ------------------------------------------------------------
# 9. قراءة البيانات الوصفية والتحقق منها
# ------------------------------------------------------------

print("\nقراءة ملف البيانات الوصفية...")

metadata = pd.read_csv(METADATA_PATH)

required_columns = [
    "image_id",
    "lesion_id",
    "dx"
]

missing_columns = [
    col for col in required_columns
    if col not in metadata.columns
]

if missing_columns:
    raise ValueError(
        f"أعمدة مطلوبة غير موجودة: {missing_columns}"
    )

metadata["dx"] = metadata["dx"].astype(str)

# ------------------------------------------------------------
# 10. فحص الصور والفئات
# ------------------------------------------------------------

all_images = (
    list(IMAGE_DIR.rglob("*.jpg")) +
    list(IMAGE_DIR.rglob("*.jpeg")) +
    list(IMAGE_DIR.rglob("*.png"))
)

print("\n" + "=" * 55)
print("ملخص تجهيز مجموعة HAM10000")
print("=" * 55)

print(f"عدد سجلات البيانات الوصفية: {len(metadata)}")
print(f"عدد الصور المستخرجة: {len(all_images)}")
print(f"عدد الآفات المختلفة: {metadata['lesion_id'].nunique()}")

print("\nتوزيع الفئات:")
print(metadata["dx"].value_counts())

print("\nالمجلد الرئيسي:", DATA_ROOT)
print("مجلد الصور:", IMAGE_DIR)
print("ملف البيانات الوصفية:", METADATA_PATH)

# ------------------------------------------------------------
# 11. حفظ نسخة من البيانات الوصفية بعد القراءة
# ------------------------------------------------------------

metadata.to_csv(
    DATA_ROOT / "metadata_verified.csv",
    index=False
)

print("\nتم تجهيز البيانات بنجاح.")
print("أصبحت البيانات جاهزة للخلايا التالية.")

## 3. قراءة البيانات وربط الصور بالتسميات

- `dx`: رمز الفئة المرضية.
- `lesion_id`: معرّف الآفة، ويُستخدم لمنع انتقال صور الآفة نفسها إلى أكثر من مجموعة.
- `image_path`: مسار الصورة.
- `label`: رقم الفئة الذي يتعامل معه النموذج.

ترتيب الفئات ثابت ويجب الحفاظ عليه عند حفظ النموذج وتشغيله.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
IMG_SIZE=224
BATCH_SIZE=32
EPOCHS_HEAD=8
EPOCHS_FINE=15
AUTOTUNE=tf.data.AUTOTUNE

CLASS_NAMES=['akiec','bcc','bkl','df','mel','nv','vasc']
ARABIC={
'akiec':'التقرّن السفعي / سرطان حرشفي موضعي',
'bcc':'سرطان الخلايا القاعدية',
'bkl':'آفات شبيهة بالتقرّن الحميد',
'df':'الورم الليفي الجلدي',
'mel':'الميلانوما (الورم الميلانيني)',
'nv':'الشامات / الوحمات الميلانينية',
'vasc':'الآفات الوعائية'
}
CLASS_TO_ID={c:i for i,c in enumerate(CLASS_NAMES)}
ID_TO_CLASS={i:c for c,i in CLASS_TO_ID.items()}

meta_files=list(DATA_ROOT.rglob('HAM10000_metadata.csv'))
if not meta_files: meta_files=list(DATA_ROOT.rglob('*metadata*.csv'))
if not meta_files: raise FileNotFoundError('لم أعثر على ملف HAM10000_metadata.csv داخل ZIP.')
df=pd.read_csv(meta_files[0])

image_files=[]
for ext in ('*.jpg','*.jpeg','*.png'): image_files += list(DATA_ROOT.rglob(ext))
image_map={p.stem:str(p) for p in image_files}
df['image_path']=df['image_id'].map(image_map)
df=df.dropna(subset=['image_path','dx','lesion_id'])
df=df[df.dx.isin(CLASS_NAMES)].copy()
df['label']=df.dx.map(CLASS_TO_ID).astype('int32')
print('عدد الصور المرتبطة:',len(df))
display(df.head())
if len(df)==0: raise RuntimeError('لم يتم ربط الصور. راجع بنية ZIP.')

counts=df.dx.value_counts().reindex(CLASS_NAMES,fill_value=0)
sns.barplot(x=counts.index,y=counts.values); plt.title('HAM10000 class distribution'); plt.show()
display(pd.DataFrame({'الفئة':CLASS_NAMES,'الاسم العربي':[ARABIC[c] for c in CLASS_NAMES],
                      'عدد الصور':[int(counts[c]) for c in CLASS_NAMES]}))

## 4. تقسيم البيانات وتحسين الصور

نستخدم `StratifiedGroupKFold` حتى نحافظ قدر الإمكان على توزيع الفئات، مع تجميع صور الآفة الواحدة في مجموعة واحدة. بعد ذلك نجهز دالة قراءة الصور وطبقات تحسين الصور.

ملاحظة: هذه الخطوة تنشئ مجموعات 224×224 الأولية أيضًا، لكن التجربة المعتمدة في هذا الدفتر هي 260×260 في الخطوة التالية.

In [ ]:
# =====================================================
# الخلية الثالثة: تقسيم البيانات وتجهيز الصور والتعزيز
# =====================================================

import tensorflow as tf
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# الإعدادات الأساسية
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# -----------------------------------------------------
# 1. تقسيم البيانات إلى تدريب وتحقق واختبار
# -----------------------------------------------------

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

folds = list(
    sgkf.split(df, y=df.label, groups=df.lesion_id)
)

test_idx = folds[0][1]
val_idx = folds[1][1]

train_idx = np.concatenate([
    folds[i][1] for i in (2, 3, 4)
])

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print('Train/Val/Test:', len(train_df), len(val_df), len(test_df))

print(
    'تداخل الآفات train-test:',
    len(set(train_df.lesion_id) & set(test_df.lesion_id))
)

# -----------------------------------------------------
# 2. تجهيز تقنيات تحسين الصور
# -----------------------------------------------------

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.12),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
    tf.keras.layers.RandomContrast(0.12)
], name='augmentation')

# -----------------------------------------------------
# 3. قراءة الصور وتجهيزها
# -----------------------------------------------------

def load_image(path, label):

    raw = tf.io.read_file(path)

    img = tf.image.decode_image(
        raw,
        channels=3,
        expand_animations=False
    )

    img.set_shape([None, None, 3])

    img = tf.image.resize(
        img,
        [IMG_SIZE, IMG_SIZE]
    )

    return tf.cast(img, tf.float32), tf.cast(label, tf.int32)


# -----------------------------------------------------
# 4. تجهيز بيانات التحقق والاختبار
# -----------------------------------------------------

def eval_ds(frame):

    ds = tf.data.Dataset.from_tensor_slices((
        frame.image_path.values,
        frame.label.values.astype('int32')
    ))

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


# -----------------------------------------------------
# 5. موازنة الفئات وتجهيز بيانات التدريب
# -----------------------------------------------------

class_streams = []

for cid in range(len(CLASS_NAMES)):

    part = train_df[train_df.label == cid]

    if len(part) == 0:
        raise ValueError(
            'فئة بلا عينات تدريب: ' + CLASS_NAMES[cid]
        )

    ds = tf.data.Dataset.from_tensor_slices((
        part.image_path.values,
        part.label.values.astype('int32')
    ))

    ds = ds.shuffle(
        max(len(part), 100),
        seed=SEED,
        reshuffle_each_iteration=True
    ).repeat()

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.map(
        lambda x, y: (augment(x, training=True), y),
        num_parallel_calls=AUTOTUNE
    )

    class_streams.append(ds)


train_ds = tf.data.Dataset.sample_from_datasets(
    class_streams,
    weights=[1 / len(CLASS_NAMES)] * len(CLASS_NAMES),
    seed=SEED
)

train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = eval_ds(val_df)
test_ds = eval_ds(test_df)

# -----------------------------------------------------
# 6. تحديد عدد خطوات التدريب
# -----------------------------------------------------

STEPS = int(np.ceil(len(train_df) / BATCH_SIZE))

print('\nتم تجهيز البيانات بنجاح.')
print('عدد صور التدريب:', len(train_df))
print('عدد صور التحقق:', len(val_df))
print('عدد صور الاختبار:', len(test_df))
print('حجم الصورة:', IMG_SIZE)
print('حجم الدفعة:', BATCH_SIZE)
print('Steps per epoch:', STEPS)

## 5. تجهيز التجربة النهائية بحجم 260×260

الإعدادات المعتمدة:
- حجم الصورة: 260×260.
- Batch size: 16.
- عدد الفئات: 7.
- تحسين الصور يطبق على التدريب فقط.
- تُوازن تدفقات التدريب باختيار الفئات بنسب متساوية.

In [ ]:
# ==========================================
# تجربة جديدة: تجهيز البيانات بحجم 260x260
# ==========================================

import tensorflow as tf
import numpy as np
import math

# إعدادات التجربة الجديدة
IMG_SIZE_260 = 260
BATCH_SIZE_260 = 16
AUTOTUNE = tf.data.AUTOTUNE

print("إعداد الصور بحجم 260x260 ...")

# ------------------------------------------
# 1. دالة تحميل الصور بالحجم الجديد
# ------------------------------------------

def load_image_260(path, label):
    raw = tf.io.read_file(path)

    img = tf.image.decode_image(
        raw,
        channels=3,
        expand_animations=False
    )

    img.set_shape([None, None, 3])

    img = tf.image.resize(
        img,
        [IMG_SIZE_260, IMG_SIZE_260]
    )

    img = tf.cast(img, tf.float32)

    return img, tf.cast(label, tf.int32)


# ------------------------------------------
# 2. تطبيق تحسين الصور على التدريب فقط
# ------------------------------------------

def load_and_augment_260(path, label):
    img, label = load_image_260(path, label)

    img = augment(img, training=True)

    return img, label


# ------------------------------------------
# 3. إنشاء مجموعات التدريب المتوازنة
# ------------------------------------------

class_streams_260 = []

for class_id in range(len(CLASS_NAMES)):

    class_df = train_df[
        train_df["label"] == class_id
    ]

    class_dataset = tf.data.Dataset.from_tensor_slices(
        (
            class_df["image_path"].values,
            class_df["label"].values
        )
    )

    class_dataset = class_dataset.shuffle(
        buffer_size=min(len(class_df), 1000),
        seed=SEED,
        reshuffle_each_iteration=True
    )

    class_dataset = class_dataset.repeat()

    class_dataset = class_dataset.map(
        load_and_augment_260,
        num_parallel_calls=AUTOTUNE
    )

    class_streams_260.append(class_dataset)


# دمج الفئات بنسب متساوية
train_ds_260 = tf.data.Dataset.sample_from_datasets(
    class_streams_260,
    weights=[1.0 / len(CLASS_NAMES)] * len(CLASS_NAMES),
    seed=SEED
)

train_ds_260 = (
    train_ds_260
    .batch(BATCH_SIZE_260)
    .prefetch(AUTOTUNE)
)


# ------------------------------------------
# 4. إنشاء بيانات التحقق
# ------------------------------------------

val_ds_260 = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].values,
        val_df["label"].values
    )
)

val_ds_260 = (
    val_ds_260
    .map(load_image_260, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE_260)
    .prefetch(AUTOTUNE)
)


# ------------------------------------------
# 5. إنشاء بيانات الاختبار
# ------------------------------------------

test_ds_260 = tf.data.Dataset.from_tensor_slices(
    (
        test_df["image_path"].values,
        test_df["label"].values
    )
)

test_ds_260 = (
    test_ds_260
    .map(load_image_260, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE_260)
    .prefetch(AUTOTUNE)
)


# ------------------------------------------
# 6. حساب خطوات التدريب
# ------------------------------------------

STEPS_260 = math.ceil(
    len(train_df) / BATCH_SIZE_260
)


# ------------------------------------------
# 7. فحص البيانات
# ------------------------------------------

images, labels = next(iter(train_ds_260))

print("\nتم تجهيز البيانات بنجاح!")
print("Train batch shape:", images.shape)
print("Labels shape:", labels.shape)
print("Validation batches:", len(val_ds_260))
print("Test batches:", len(test_ds_260))
print("Training steps:", STEPS_260)
print("Image size:", IMG_SIZE_260)
print("Batch size:", BATCH_SIZE_260)

## 6. بناء EfficientNetB0

نستخدم أوزان ImageNet كبداية (Transfer Learning)، ثم نضيف طبقات تجميع وتصنيف تناسب الفئات السبع. في البداية نجمّد القاعدة الأساسية وندرّب رأس التصنيف فقط.

In [ ]:
# ==========================================
# بناء EfficientNetB0 بحجم 260x260
# ==========================================

import tensorflow as tf

# بناء النموذج الأساسي
base_260 = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(260, 260, 3)
)

# تجميد النموذج الأساسي مبدئياً
base_260.trainable = False

# طبقات التصنيف
inputs_260 = tf.keras.Input(
    shape=(260, 260, 3)
)

x = base_260(inputs_260, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dropout(0.35)(x)

outputs_260 = tf.keras.layers.Dense(
    len(CLASS_NAMES),
    activation="softmax"
)(x)

model_260 = tf.keras.Model(
    inputs=inputs_260,
    outputs=outputs_260
)

# عرض ملخص النموذج
model_260.summary()

print("\nتم بناء النموذج الجديد بنجاح.")

## 7. التدريب الأولي باستخدام Focal Loss

تساعد Focal Loss على إعطاء اهتمام أكبر للأمثلة التي يصعب على النموذج تصنيفها. تحفظ أفضل نسخة من مرحلة التدريب الأولي في Google Drive.

In [ ]:
import tensorflow as tf

def focal_loss_260(gamma=1.5):
    def loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        indices = tf.stack([tf.range(tf.shape(y_true)[0]), y_true], axis=1)
        pt = tf.gather_nd(y_pred, indices)
        focal_factor = tf.pow(1.0 - pt, gamma)
        return -focal_factor * tf.math.log(pt)
    return loss

model_260.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-5
    ),
    loss=focal_loss_260(gamma=1.5),
    metrics=["accuracy"]
)

HEAD_CHECKPOINT = os.path.join(SAVE_DIR, "best_260_head.keras")

callbacks_260 = [
    tf.keras.callbacks.ModelCheckpoint(
        HEAD_CHECKPOINT,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

history_head_260 = model_260.fit(
    train_ds_260,
    steps_per_epoch=STEPS_260,
    validation_data=val_ds_260,
    epochs=10,
    callbacks=callbacks_260
)

## 8. Fine-Tuning

بعد تدريب رأس التصنيف، نفتح آخر 40 طبقة من EfficientNetB0 للتدريب، مع إبقاء طبقات BatchNormalization مجمّدة، ونستخدم معدل تعلم أصغر حتى تكون التعديلات على الأوزان تدريجية.

**مهم:** أفضل أوزان Fine-Tuning تحفظ مباشرة في Google Drive.

In [ ]:
base_260.trainable = True

for layer in base_260.layers[:-40]:
    layer.trainable = False

for layer in base_260.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_layers = sum(1 for layer in base_260.layers if layer.trainable)
print("Trainable layers:", trainable_layers)

model_260.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-5,
        weight_decay=1e-5
    ),
    loss=focal_loss_260(gamma=1.5),
    metrics=["accuracy"]
)

FINETUNE_CHECKPOINT = os.path.join(
    SAVE_DIR, "best_260_finetuned.weights.h5"
)

callbacks_finetune_260 = [
    tf.keras.callbacks.ModelCheckpoint(
        FINETUNE_CHECKPOINT,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

history_finetune_260 = model_260.fit(
    train_ds_260,
    steps_per_epoch=STEPS_260,
    validation_data=val_ds_260,
    epochs=15,
    callbacks=callbacks_finetune_260
)

# استرجاع أفضل أوزان Fine-Tuning المسجلة
model_260.load_weights(FINETUNE_CHECKPOINT)
print("تم تحميل أفضل أوزان Fine-Tuning.")

## 9. تقييم الاختبار النهائي باستخدام TTA

تُجرى أربعة تنبؤات لكل صورة: الأصلية، ثم تدوير 90 و180 و270 درجة. بعد ذلك نحسب متوسط الاحتمالات ونستخرج الفئة النهائية.

لا تستخدمي مجموعة الاختبار أثناء اختيار الإعدادات أو تحسين التدريب؛ فهي مخصصة للتقييم النهائي.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

test_predictions = []

for images, labels in test_ds_260:
    p0 = model_260(images, training=False).numpy()
    p90 = model_260(tf.image.rot90(images, k=1), training=False).numpy()
    p180 = model_260(tf.image.rot90(images, k=2), training=False).numpy()
    p270 = model_260(tf.image.rot90(images, k=3), training=False).numpy()
    test_predictions.append((p0 + p90 + p180 + p270) / 4.0)

test_prob_final = np.concatenate(test_predictions, axis=0)
test_pred_final = np.argmax(test_prob_final, axis=1)
test_true_final = test_df["label"].to_numpy()

test_accuracy = accuracy_score(test_true_final, test_pred_final)
test_macro_f1 = f1_score(test_true_final, test_pred_final, average="macro")

print("Test Accuracy:", round(test_accuracy, 4))
print("Test Macro F1:", round(test_macro_f1, 4))

print("\nClassification Report:\n")
print(classification_report(
    test_true_final,
    test_pred_final,
    labels=range(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

cm_final = confusion_matrix(
    test_true_final,
    test_pred_final,
    labels=range(len(CLASS_NAMES))
)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm_final, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final Test Confusion Matrix - EfficientNetB0 260 + TTA")
plt.tight_layout()
plt.show()

## 10. حفظ النموذج والنتائج والتوثيق في Google Drive

هذه الخلية تحفظ:
- النموذج بصيغة `.keras`.
- ترتيب أسماء الفئات في JSON.
- نتائج الاختبار في JSON.
- ملف توثيق عربي مختصر.
- حزمة ZIP تضم الملفات الأساسية.

شغّليها بعد انتهاء التقييم النهائي.

In [ ]:
import json
import zipfile
from datetime import datetime

MODEL_PATH = os.path.join(SAVE_DIR, "skin_lesion_efficientnetb0_260.keras")
CLASS_PATH = os.path.join(SAVE_DIR, "class_names.json")
METRICS_PATH = os.path.join(SAVE_DIR, "evaluation_results.json")
README_PATH = os.path.join(SAVE_DIR, "توثيق_المشروع.md")
ZIP_PATH = os.path.join(SAVE_DIR, "skin_lesion_model_package.zip")

# حفظ النموذج
model_260.save(MODEL_PATH)

# حفظ ترتيب الفئات
with open(CLASS_PATH, "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

# حفظ المقاييس الفعلية المحسوبة في هذه الجلسة
metrics = {
    "model": "EfficientNetB0",
    "dataset": "HAM10000",
    "image_size": [260, 260],
    "number_of_classes": len(CLASS_NAMES),
    "test_images": int(len(test_true_final)),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "evaluation_method": "TTA: original + rotations 90, 180, 270",
    "class_names": CLASS_NAMES,
    "saved_at": datetime.now().isoformat(timespec="seconds")
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# توثيق مختصر مبني على إعدادات هذا الدفتر ونتائجه الفعلية
documentation = f'''
# مشروع تصنيف أمراض الجلد باستخدام HAM10000 وEfficientNetB0

## فكرة المشروع
تصنيف صور الآفات الجلدية إلى سبع فئات باستخدام التعلم العميق.

## البيانات
HAM10000. عدد الصور المستخدمة في التقسيم: {len(df)}.

## الفئات
{", ".join(CLASS_NAMES)}

## النموذج
EfficientNetB0 مع Transfer Learning ثم Fine-Tuning.
حجم الإدخال: 260×260.
حجم الدفعة: 16.
دالة الخسارة: Focal Loss (gamma=1.5).

## تقسيم البيانات
استخدم StratifiedGroupKFold مع lesion_id لتقليل تسرب صور الآفة نفسها بين المجموعات.

## التقييم
Test Accuracy: {test_accuracy:.4f}
Test Macro F1: {test_macro_f1:.4f}
عدد صور الاختبار: {len(test_true_final)}

استخدم TTA بأربع وضعيات: الصورة الأصلية والتدوير 90 و180 و270 درجة.

## الملفات
- skin_lesion_efficientnetb0_260.keras: النموذج النهائي.
- class_names.json: ترتيب الفئات.
- evaluation_results.json: نتائج التقييم.
- skin_lesion_model_package.zip: حزمة النموذج والفئات.

## تنبيه
المشروع تعليمي وبحثي، وليس أداة تشخيص طبي أو بديلاً عن الطبيب المختص.
'''

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(documentation)

# إنشاء ZIP
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH, arcname=os.path.basename(MODEL_PATH))
    z.write(CLASS_PATH, arcname=os.path.basename(CLASS_PATH))
    z.write(METRICS_PATH, arcname=os.path.basename(METRICS_PATH))
    z.write(README_PATH, arcname=os.path.basename(README_PATH))

print("اكتمل الحفظ في Google Drive:")
for p in [MODEL_PATH, CLASS_PATH, METRICS_PATH, README_PATH, ZIP_PATH]:
    print(p, "|", round(os.path.getsize(p) / (1024 * 1024), 2), "MB")

## 11. ملخص المشروع للمناقشة

### ما الذي يفعله المشروع؟
يستقبل صورة آفة جلدية، يجهزها بالحجم الذي تدرب عليه النموذج، ثم يستخدم EfficientNetB0 لإنتاج احتمالات الفئات السبع ويعرض الفئة الأعلى احتمالًا.

### المصطلحات الأساسية
- **HAM10000:** مجموعة بيانات صور آفات جلدية مع تسميات.
- **CNN:** شبكة عصبية التفافية لاستخراج السمات من الصور.
- **EfficientNetB0:** بنية CNN مستخدمة كنموذج أساسي.
- **Transfer Learning:** الاستفادة من أوزان مدربة مسبقًا.
- **Fine-Tuning:** فتح بعض الطبقات وتحديث أوزانها على بيانات المشروع.
- **Focal Loss:** دالة خسارة تركز بدرجة أكبر على الأمثلة الأصعب.
- **Data Augmentation:** إجراء تحويلات على صور التدريب لزيادة تنوعها.
- **TTA:** إجراء عدة تنبؤات بتحويلات للصورة ثم متوسط الاحتمالات.
- **Accuracy:** نسبة التنبؤات الصحيحة إجمالًا.
- **Macro F1:** متوسط F1 للفئات بالتساوي، لذلك يوضح أداء الفئات الأقل عددًا.

### ملاحظة مهمة عن النتائج
لا تكتبي نتائج سابقة ثابتة في التقرير قبل إعادة تشغيل هذا الدفتر؛ اعتمدي القيم التي ستظهر في خلية الاختبار النهائي وتحفظ في `evaluation_results.json`. النتائج السابقة المسجلة في النسخة القديمة كانت: Validation دون TTA = 73.61% Accuracy و59.76% Macro F1، وValidation مع TTA = 74.76% و61.09%، وTest مع TTA = 73.83% و54.52%. هذه نتائج تاريخية من تشغيل سابق وليست ضمانًا لتكرارها.

### قائمة التحقق
- [ ] تفعيل GPU.
- [ ] التأكد من اكتمال تنزيل HAM10000.
- [ ] تشغيل الخلايا بالترتيب.
- [ ] التأكد من عدم وجود تداخل في `lesion_id` بين المجموعات.
- [ ] التأكد من حفظ أفضل أوزان Fine-Tuning في Drive.
- [ ] حفظ النموذج والنتائج والحزمة النهائية.
- [ ] تجربة تحميل النموذج من الحزمة قبل التسليم.